# Dataset 3 — v1 Embeddings (Node2Vec, unbiased)

In [1]:
import sys
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.models.embeddings import Node2VecConfig, extract_embeddings

EMB_ROOT = PROJECT_ROOT / 'src' / 'data' / 'embeddings'
pd.set_option('display.max_columns', 200)
print(f'Project root: {PROJECT_ROOT}')

def emb_path(name):
    name = str(name)
    ds  = 'dataset_3' if 'dataset3' in name else ('dataset_2' if 'dataset2' in name else 'dataset_1')
    sub = 'network_based' if name.startswith('node2vec') else 'feature_based'
    return EMB_ROOT / ds / sub / name

DATASET_DIR = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_3'
N = 10000

e = pd.read_csv(DATASET_DIR / 'edges.csv')
edges = e.rename(columns={'from': 'Sourceid', 'to': 'Targetid', 'weight': 'Weights'})[['Sourceid', 'Targetid', 'Weights']]
import numpy as np
nodes = pd.DataFrame({'index': np.arange(1, N + 1), 'feat': 1.0})   # dummy feat (Node2Vec ignores it)
target = pd.read_csv(DATASET_DIR / 'targets' / 'target.csv')
target_cols = ['log_systemic_risk_label']   # log-only, matching Dataset 1

def build_dataset(config, output_path):
    emb, _ = extract_embeddings(edges, nodes, config)
    emb_cols = [c for c in emb.columns if c.startswith('emb_')]
    merged = emb[['bank_id'] + emb_cols].merge(target, on='bank_id', how='inner')[['bank_id'] + emb_cols + target_cols]
    merged.to_parquet(output_path, index=False)
    return merged


Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


In [2]:
TARGET_COL = 'log_systemic_risk_label'

cfg_node2vec_v1_32   = Node2VecConfig(embedding_dim=32,  walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=1, batch_size=128, lr=0.01, epochs=100, device='cpu')
cfg_node2vec_v1_64   = Node2VecConfig(embedding_dim=64,  walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=1, batch_size=128, lr=0.01, epochs=100, device='cpu')
cfg_node2vec_v1_128  = Node2VecConfig(embedding_dim=128, walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=5, batch_size=128, lr=0.01, epochs=100, device='cpu')

OUTPUTS = {
    'node2vec_v1_32':   emb_path('node2vec_v1_32_dataset3_dataset.parquet'),
    'node2vec_v1_64':   emb_path('node2vec_v1_64_dataset3_dataset.parquet'),
    'node2vec_v1_128':  emb_path('node2vec_v1_128_dataset3_dataset.parquet'),
}
OUTPUTS

{'node2vec_v1_32': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/dataset_3/network_based/node2vec_v1_32_dataset3_dataset.parquet'),
 'node2vec_v1_64': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/dataset_3/network_based/node2vec_v1_64_dataset3_dataset.parquet'),
 'node2vec_v1_128': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/dataset_3/network_based/node2vec_v1_128_dataset3_dataset.parquet')}

## Node2Vec v1

In [3]:
for cfg, key in [
    (cfg_node2vec_v1_32,  'node2vec_v1_32'),
    (cfg_node2vec_v1_64,  'node2vec_v1_64'),
    (cfg_node2vec_v1_128, 'node2vec_v1_128'),
]:
    df = build_dataset(cfg, OUTPUTS[key])
    emb_cols = [c for c in df.columns if c.startswith('emb_')]
    print(f'{key:24s}  shape={df.shape}  emb_cols={len(emb_cols)}')

node2vec_v1_32            shape=(10000, 34)  emb_cols=32
node2vec_v1_64            shape=(10000, 66)  emb_cols=64
node2vec_v1_128           shape=(10000, 130)  emb_cols=128


## Output

In [4]:
for name, path in OUTPUTS.items():
    print(f'{name:24s} -> {path.name}')

node2vec_v1_32           -> node2vec_v1_32_dataset3_dataset.parquet
node2vec_v1_64           -> node2vec_v1_64_dataset3_dataset.parquet
node2vec_v1_128          -> node2vec_v1_128_dataset3_dataset.parquet
